In [1]:
import os
import glob
import sys

import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr5"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
    cell_types = ["c1", "c2", "c4", "c6", "c17"]
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
    cell_types = ["c1", "c2", "c4", "c6", "c13", "c17"]
else:
    raise ValueError

In [5]:
PATHS_FROM_GENALG = f'../../generator/genetic_alg/results/{utr_type}/*.csv'
PATHS_FROM_DIFFUSION = f'../../generator/diffusion/generate/results/*{utr_type[-1]}UTR.csv'

In [6]:
import re

paths_genalg = glob.glob(PATHS_FROM_GENALG)
read_dfs_ga = []
for file in paths_genalg:
    bn = os.path.basename(file)
    ct_pair = re.match(r"utr\d-seed=\d+-genes=\d+-(c\d+-c\d+).csv", bn).group(1)
    df_chunk = pd.read_csv(file, header=0, usecols=["num", "seq"])
    df_chunk["generation_type"] = "genetic_algorithm"
    df_chunk["ct_pair"] = ct_pair
    read_dfs_ga.append(df_chunk)
df_ga = pd.concat(read_dfs_ga, axis=0)

In [7]:
paths_diffusion = glob.glob(PATHS_FROM_DIFFUSION)
read_dfs_dif = []
for file in paths_diffusion:
    bn = os.path.basename(file)
    ct_pair = re.match(r"cell_(c\d+)_epoch_\d+_\dUTR.csv", bn).group(1)
    df_chunk = pd.read_csv(file, header=0, usecols=["seq", "mass_center"])
    df_chunk.rename({"mass_center": "query_exp"}, axis=1, inplace=True)
    df_chunk["generation_type"] = "diffusion"
    df_chunk["ct_pair"] = ct_pair
    read_dfs_dif.append(df_chunk)
df_dif = pd.concat(read_dfs_dif, axis=0)

In [8]:
df = pd.concat([df_ga, df_dif]).drop("num", axis=1)
df

,seq,generation_type,ct_pair,query_exp
0,GACCACCGGGAGAGGACGAAGAAGAGGAAGAAGAAGAAGAAGAACC...,genetic_algorithm,c1-c2,NaN
1,GTCAAGGCTCCGACTCGGAGACCGAATGAAAGCACCGGACAGAAGA...,genetic_algorithm,c1-c2,NaN
2,GAGGAGACGGAGAGGATAAAACTAAAGAAGTCGGCCACGCAGACGA...,genetic_algorithm,c1-c2,NaN
3,TTCGCAGGGCGAGTGAAGAAGGAAGAATGGAGGAGTACCGCGCATA...,genetic_algorithm,c1-c2,NaN
4,GAGTAGCGGAAGAATACGAGTAGAGACGAGGCGAGTACGAGCGCGA...,genetic_algorithm,c1-c2,NaN
...,...,...,...,...
4091,GCTCTTTTCATTTTTTTCTTTTTTTCTCTTCTTCTTTTAATCTTCT...,diffusion,c1,2.388341
4092,GAGAGAGAGGAATAGAGTTGTGTGTGAGGTTGGTTGAGTGAAGGAG...,diffusion,c1,2.105134
4093,AAACAACCAACACAACAACCAAAACCAAAAACACACACAAACAACA...,diffusion,c1,2.238525
4094,ACTCCCGTCCTGCCAGCTGGCAGCGGCAATCAAAGTCACAGAAGGA...,diffusion,c1,2.880534


In [9]:
df.drop_duplicates(subset=["seq", "generation_type", "ct_pair"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,seq,generation_type,ct_pair,query_exp
0,GACCACCGGGAGAGGACGAAGAAGAGGAAGAAGAAGAAGAAGAACC...,genetic_algorithm,c1-c2,NaN
1,GTCAAGGCTCCGACTCGGAGACCGAATGAAAGCACCGGACAGAAGA...,genetic_algorithm,c1-c2,NaN
2,GAGGAGACGGAGAGGATAAAACTAAAGAAGTCGGCCACGCAGACGA...,genetic_algorithm,c1-c2,NaN
3,TTCGCAGGGCGAGTGAAGAAGGAAGAATGGAGGAGTACCGCGCATA...,genetic_algorithm,c1-c2,NaN
4,GAGTAGCGGAAGAATACGAGTAGAGACGAGGCGAGTACGAGCGCGA...,genetic_algorithm,c1-c2,NaN
...,...,...,...,...
5073889,GCTCTTTTCATTTTTTTCTTTTTTTCTCTTCTTCTTTTAATCTTCT...,diffusion,c1,2.388341
5073890,GAGAGAGAGGAATAGAGTTGTGTGTGAGGTTGGTTGAGTGAAGGAG...,diffusion,c1,2.105134
5073891,AAACAACCAACACAACAACCAAAACCAAAAACACACACAAACAACA...,diffusion,c1,2.238525
5073892,ACTCCCGTCCTGCCAGCTGGCAGCGGCAATCAAAGTCACAGAAGGA...,diffusion,c1,2.880534


In [10]:
df["seq"].duplicated().value_counts()

seq
False    5073720
True         174
Name: count, dtype: int64

In [11]:
df.drop_duplicates(subset=["seq"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,seq,generation_type,ct_pair,query_exp
0,GACCACCGGGAGAGGACGAAGAAGAGGAAGAAGAAGAAGAAGAACC...,genetic_algorithm,c1-c2,NaN
1,GTCAAGGCTCCGACTCGGAGACCGAATGAAAGCACCGGACAGAAGA...,genetic_algorithm,c1-c2,NaN
2,GAGGAGACGGAGAGGATAAAACTAAAGAAGTCGGCCACGCAGACGA...,genetic_algorithm,c1-c2,NaN
3,TTCGCAGGGCGAGTGAAGAAGGAAGAATGGAGGAGTACCGCGCATA...,genetic_algorithm,c1-c2,NaN
4,GAGTAGCGGAAGAATACGAGTAGAGACGAGGCGAGTACGAGCGCGA...,genetic_algorithm,c1-c2,NaN
...,...,...,...,...
5073715,GCTCTTTTCATTTTTTTCTTTTTTTCTCTTCTTCTTTTAATCTTCT...,diffusion,c1,2.388341
5073716,GAGAGAGAGGAATAGAGTTGTGTGTGAGGTTGGTTGAGTGAAGGAG...,diffusion,c1,2.105134
5073717,AAACAACCAACACAACAACCAAAACCAAAAACACACACAAACAACA...,diffusion,c1,2.238525
5073718,ACTCCCGTCCTGCCAGCTGGCAGCGGCAATCAAAGTCACAGAAGGA...,diffusion,c1,2.880534


In [12]:
df["seq"].duplicated().value_counts()

seq
False    5073720
Name: count, dtype: int64

In [13]:
current_cols = df.columns
current_cols

Index(['seq', 'generation_type', 'ct_pair', 'query_exp'], dtype='object')

In [14]:
for ct in cell_types:
    df[ct] = 0
df = pd.melt(df, id_vars=current_cols, value_vars=cell_types, var_name="cell_type", value_name="value").drop("value", axis=1)

In [15]:
df.sort_values(by=["ct_pair", "seq", "cell_type"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [16]:
df.head(10)

,seq,generation_type,ct_pair,query_exp,cell_type
0,AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAAC...,diffusion,c1,2.743273,c1
1,AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAAC...,diffusion,c1,2.743273,c17
2,AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAAC...,diffusion,c1,2.743273,c2
3,AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAAC...,diffusion,c1,2.743273,c4
4,AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAAC...,diffusion,c1,2.743273,c6
5,AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAA...,diffusion,c1,2.384502,c1
6,AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAA...,diffusion,c1,2.384502,c17
7,AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAA...,diffusion,c1,2.384502,c2
8,AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAA...,diffusion,c1,2.384502,c4
9,AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAA...,diffusion,c1,2.384502,c6


In [17]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

5

In [18]:
batch_size = 1024

In [19]:
num_workers = 32

In [20]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [21]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [22]:
ckpt = torch.load(MODEL_PATH)
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding

loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

In [25]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[1],
    deterministic=True,
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [26]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [27]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")
seq_embedding

array([[ 5.31960875e-02,  2.20255315e-01,  1.00871073e-02, ...,
         1.10093683e-01,  3.04780602e-01, -5.39521500e-02],
       [ 6.64573908e-02,  1.94856510e-01,  3.30592692e-02, ...,
         1.11877836e-01,  2.34562501e-01, -8.31770152e-02],
       [ 6.30080327e-02,  2.10280001e-01,  1.46588078e-04, ...,
         1.27121896e-01,  2.64979213e-01, -7.56667629e-02],
       ...,
       [ 1.34117082e-01,  2.07183152e-01,  8.54923204e-02, ...,
         1.57462806e-01,  1.92733258e-01,  2.20610499e-01],
       [ 1.18487164e-01,  2.38783941e-01,  7.04548061e-02, ...,
         1.84970215e-01,  1.93883657e-01,  2.79327720e-01],
       [ 1.20051846e-01,  2.10240707e-01,  1.05024241e-01, ...,
         1.55480713e-01,  1.59082130e-01,  3.07532012e-01]], dtype=float32)

In [28]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)
pred_mass_center

array([[1.8326339, 1.8542134, 2.2514548, 1.9718552, 1.7936332],
       [1.7961495, 1.8503406, 2.1894696, 1.9655467, 1.7564154],
       [1.8289535, 1.865437 , 2.225205 , 1.933079 , 1.7781668],
       ...,
       [2.760158 , 2.7919297, 2.6355278, 2.6546104, 2.798068 ],
       [2.7479825, 2.847245 , 2.6305676, 2.6686156, 2.8334386],
       [2.7646105, 2.8870604, 2.5829308, 2.6247194, 2.8402045]],
      dtype=float32)

In [29]:
result = pd.DataFrame(
    pred_mass_center,
    columns=pd.MultiIndex.from_product([["pred_mass_center"], cell_types]),
    index=df.iloc[::num_classes]["seq"]
)
result["pred_mass_center"].head()

,c1,c2,c4,c6,c17
seq,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAACACAG,1.832634,1.854213,2.251455,1.971855,1.793633
AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAATTTT,1.796149,1.850341,2.189470,1.965547,1.756415
AAAAAAAAAAAAAAAAAGAAAATAAAAAAAAAAAAAAAAAAAATAAATTT,1.828954,1.865437,2.225205,1.933079,1.778167
AAAAAAAAAAAAAAATTTCATTTTTTTAAAAAAACAAAAAAAAAAAAAAA,1.868739,1.838129,2.235678,1.947978,1.737980
AAAAAAAAAACAACAAACACCCAACACTACTCTACCAAAAACCCAAAAAA,2.090951,2.078767,2.451016,2.188013,2.072864


In [30]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["pred_mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [31]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c17_022,c17_023,c17_024,c17_025,c17_026,c17_027,c17_028,c17_029,c17_030,c17_031
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAACACAG,0.053196,0.220255,0.010087,0.056278,0.241045,0.020189,-0.067779,0.274832,0.040887,-0.000588,...,-0.095859,0.046259,0.009147,0.275403,0.087716,0.437900,0.143174,0.110094,0.304781,-0.053952
AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAATTTT,0.066457,0.194857,0.033059,0.063162,0.232200,-0.016365,-0.067304,0.280030,0.051282,-0.005548,...,-0.103269,0.098289,0.028797,0.245779,0.088335,0.475380,0.160255,0.111878,0.234563,-0.083177
AAAAAAAAAAAAAAAAAGAAAATAAAAAAAAAAAAAAAAAAAATAAATTT,0.063008,0.210280,0.000147,0.055583,0.223237,0.011878,-0.069839,0.275849,0.027304,-0.011916,...,-0.098670,0.066341,0.025150,0.268850,0.085901,0.474913,0.149490,0.127122,0.264979,-0.075667
AAAAAAAAAAAAAAATTTCATTTTTTTAAAAAAACAAAAAAAAAAAAAAA,0.079194,0.201714,0.049024,0.054724,0.219200,0.045313,-0.037229,0.274284,0.052911,0.020758,...,-0.107302,0.087800,0.032456,0.202561,0.095280,0.388717,0.180386,0.088858,0.233503,-0.039796
AAAAAAAAAACAACAAACACCCAACACTACTCTACCAAAAACCCAAAAAA,0.083871,0.177492,0.033273,0.045724,0.202516,0.087214,0.013236,0.173093,0.081042,0.075949,...,0.019925,0.047933,0.050859,0.207247,0.102011,0.111962,0.199757,0.059886,0.266108,0.062671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTGAGAAGGACGCTAACGCCGAGAAGAAGCTGACGACGACTGCAGCA,0.129196,0.229216,0.061667,0.227374,0.131424,0.172324,0.156474,0.096516,0.126754,0.161680,...,0.659291,0.085067,0.146014,0.070736,0.115873,0.132874,0.084999,0.194070,0.159150,0.294947
TTTTTGAGGGGGGGGGCGAAGCCAGCAAGAAACGTGAGCAGCTGCCGATC,0.145363,0.172389,0.113262,0.179207,0.142317,0.148652,0.146352,0.140941,0.149952,0.164582,...,0.170645,0.149646,0.141770,0.112288,0.177515,0.037317,0.146369,0.142046,0.176188,0.173249
TTTTTGAGGTAGGCACCGAAGATAGGAGCACCGAGAACTCGTTGAGGATA,0.134117,0.207183,0.085492,0.176590,0.142256,0.159091,0.128629,0.129371,0.136971,0.164375,...,0.359799,0.112570,0.148332,0.093168,0.159959,0.136614,0.126739,0.157463,0.192733,0.220610


### Counting k-mers

In [32]:
sys.path.append("../../benchmark/kmers/")
from kmer_model import KmerLinearRegressor

In [33]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [34]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAACACAG,46,3,1,0,42,3,1,0,3,0,...,0,0,0,0,0,0,0,0,0,0
AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAATTTT,42,0,0,8,37,0,0,5,0,0,...,0,0,0,0,0,0,0,0,0,2
AAAAAAAAAAAAAAAAAGAAAATAAAAAAAAAAAAAAAAAAAATAAATTT,44,0,1,5,40,0,1,3,0,0,...,0,0,0,0,0,0,0,0,0,1
AAAAAAAAAAAAAAATTTCATTTTTTTAAAAAAACAAAAAAAAAAAAAAA,38,2,0,10,34,1,0,2,2,0,...,0,0,0,0,0,0,1,1,0,6
AAAAAAAAAACAACAAACACCCAACACTACTCTACCAAAAACCCAAAAAA,32,15,0,3,22,9,0,0,7,5,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTGAGAAGGACGCTAACGCCGAGAAGAAGCTGACGACGACTGCAGCA,16,11,15,8,4,5,6,0,2,1,...,0,0,2,1,0,0,0,0,1,3
TTTTTGAGGGGGGGGGCGAAGCCAGCAAGAAACGTGAGCAGCTGCCGATC,12,10,20,8,4,1,6,1,3,2,...,0,0,2,1,0,0,0,0,1,3
TTTTTGAGGTAGGCACCGAAGATAGGAGCACCGAGAACTCGTTGAGGATA,15,8,16,11,2,3,7,2,2,2,...,1,0,2,0,0,0,0,0,2,3


### Adding final info

In [35]:
dfseq = df.set_index("seq").iloc[::num_classes]
result["ct_pair"] = dfseq["ct_pair"]
result["generation_type"] = dfseq["generation_type"]
result["query_exp"] = dfseq["query_exp"]

In [36]:
result

pred_mass_center            \
                                                                 c1        c2   
seq                                                                             
AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAACACAG         1.832634  1.854213   
AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAATTTT         1.796149  1.850341   
AAAAAAAAAAAAAAAAAGAAAATAAAAAAAAAAAAAAAAAAAATAAATTT         1.828954  1.865437   
AAAAAAAAAAAAAAATTTCATTTTTTTAAAAAAACAAAAAAAAAAAAAAA         1.868739  1.838129   
AAAAAAAAAACAACAAACACCCAACACTACTCTACCAAAAACCCAAAAAA         2.090951  2.078767   
...                                                             ...       ...   
TTTTTGAGAAGGACGCTAACGCCGAGAAGAAGCTGACGACGACTGCAGCA         2.828280  2.789872   
TTTTTGAGGGGGGGGGCGAAGCCAGCAAGAAACGTGAGCAGCTGCCGATC         2.637582  2.656020   
TTTTTGAGGTAGGCACCGAAGATAGGAGCACCGAGAACTCGTTGAGGATA         2.760158  2.791930   
TTTTTGGAGAAGTTATTCACAAGACGAGACCGAAGAAAGACACGAGCAAC         2.747983  2.847245   
TTTTTGTGGAAGACGGATCGCACCCCGAGAACGGAACGGTCAGAAGGCCA         2.764611  2.887060   

                                                                        \
                                                          c4        c6   
seq                                                                      
AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAACACAG  2.251455  1.971855   
AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAATTTT  2.189470  1.965547   
AAAAAAAAAAAAAAAAAGAAAATAAAAAAAAAAAAAAAAAAAATAAATTT  2.225205  1.933079   
AAAAAAAAAAAAAAATTTCATTTTTTTAAAAAAACAAAAAAAAAAAAAAA  2.235678  1.947978   
AAAAAAAAAACAACAAACACCCAACACTACTCTACCAAAAACCCAAAAAA  2.451016  2.188013   
...                                                      ...       ...   
TTTTTGAGAAGGACGCTAACGCCGAGAAGAAGCTGACGACGACTGCAGCA  2.599396  2.599168   
TTTTTGAGGGGGGGGGCGAAGCCAGCAAGAAACGTGAGCAGCTGCCGATC  2.555557  2.559100   
TTTTTGAGGTAGGCACCGAAGATAGGAGCACCGAGAACTCGTTGAGGATA  2.635528  2.654610   
TTTTTGGAGAAGTTATTCACAAGACGAGACCGAAGAAAGACACGAGCAAC  2.630568  2.668616   
TTTTTGTGGAAGACGGATCGCACCCCGAGAACGGAACGGTCAGAAGGCCA  2.582931  2.624719   

                                                             embedding  \
                                                         c17    c1_000   
seq                                                                      
AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAACACAG  1.793633  0.053196   
AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAATTTT  1.756415  0.066457   
AAAAAAAAAAAAAAAAAGAAAATAAAAAAAAAAAAAAAAAAAATAAATTT  1.778167  0.063008   
AAAAAAAAAAAAAAATTTCATTTTTTTAAAAAAACAAAAAAAAAAAAAAA  1.737980  0.079194   
AAAAAAAAAACAACAAACACCCAACACTACTCTACCAAAAACCCAAAAAA  2.072864  0.083871   
...                                                      ...       ...   
TTTTTGAGAAGGACGCTAACGCCGAGAAGAAGCTGACGACGACTGCAGCA  2.910485  0.129196   
TTTTTGAGGGGGGGGGCGAAGCCAGCAAGAAACGTGAGCAGCTGCCGATC  2.619114  0.145363   
TTTTTGAGGTAGGCACCGAAGATAGGAGCACCGAGAACTCGTTGAGGATA  2.798068  0.134117   
TTTTTGGAGAAGTTATTCACAAGACGAGACCGAAGAAAGACACGAGCAAC  2.833439  0.118487   
TTTTTGTGGAAGACGGATCGCACCCCGAGAACGGAACGGTCAGAAGGCCA  2.840204  0.120052   

                                                                        \
                                                      c1_001    c1_002   
seq                                                                      
AAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAAAAAAAAAAAACACAG  0.220255  0.010087   
AAAAAAAAAAAAAAAAAAAAAAATAAATATAAAAAAAATAAAAAAATTTT  0.194857  0.033059   
AAAAAAAAAAAAAAAAAGAAAATAAAAAAAAAAAAAAAAAAAATAAATTT  0.210280  0.000147   
AAAAAAAAAAAAAAATTTCATTTTTTTAAAAAAACAAAAAAAAAAAAAAA  0.201714  0.049024   
AAAAAAAAAACAACAAACACCCAACACTACTCTACCAAAAACCCAAAAAA  0.177492  0.033273   
...                                                      ...       ...   
TTTTTGAGAAGGACGCTAACGCCGAGAAGAAGCTGACGACGACTGCAGCA  0.229216  0.061667   
TTTTTGAGGGGGGGGGCGAAGCCAGCAAGAAACGTGAGCAGCTGCCGATC  0.172389  0.113262   
TTTTTGAGGTAGGCACCGAAGATAGGAG

In [37]:
result.to_parquet(f"embeddings_mapperless_{utr_type}_generated.parquet")

---